# Build Training Data (Dev)

> **Status:** Exploratory — superseded by `notebooks/production/build_training_data_prod.ipynb`.
> This was the initial prototype for the feature engineering pipeline, built against synthetic dummy data (not real production invoice data). The join logic and feature derivation here served as the blueprint for the production version.

Loads raw generated data from `data/processed/`, derives behavioral features, enriches with census data, and assembles the final flat training table for the ML model.

## Step 1: Load Raw Data
Loads all raw CSVs generated by `generate_dummy_data.ipynb`.

In [1]:
import pandas as pd
import numpy as np

data_dir = '../../data/processed'

persons_df = pd.read_csv(f'{data_dir}/dummy_persons.csv')
addresses_df = pd.read_csv(f'{data_dir}/dummy_addresses.csv', dtype={'postal_code': str})
invoices_df = pd.read_csv(f'{data_dir}/dummy_invoices.csv')
payment_log_df = pd.read_csv(f'{data_dir}/dummy_payment_log.csv')
invoice_payments_df = pd.read_csv(f'{data_dir}/dummy_invoice_payments.csv')
calendar_events_df = pd.read_csv(f'{data_dir}/dummy_calendar_events.csv')

print(f"Persons: {len(persons_df)} rows")
print(f"Addresses: {len(addresses_df)} rows")
print(f"Invoices: {len(invoices_df)} rows")
print(f"Payment logs: {len(payment_log_df)} rows")
print(f"Invoice payments: {len(invoice_payments_df)} rows")
print(f"Calendar events: {len(calendar_events_df)} rows")

Persons: 250 rows
Addresses: 250 rows
Invoices: 4000 rows
Payment logs: 2913 rows
Invoice payments: 2913 rows
Calendar events: 3017 rows


## Step 2: Derive Person-Level Features
Derives behavioral features from the transactional data:

- **`appointment_reliability_score`**: ratio of `COMPLETED`/`CONFIRMED` appointments to total appointments (from `calendars_events`)
- **`average_days_to_pay`**: mean days between invoice `createdat` and the earliest `payment_log.recordedat` for that invoice, joined via `invoice_payments`

Persons with no payment history receive `-1` for `average_days_to_pay`.

In [2]:
# --- Derive appointment_reliability_score ---
# "Reliable" statuses: COMPLETED, CONFIRMED
# "Unreliable" statuses: CANCELED, NO_SHOW
# UNCONFIRMED/ATTEMPTED are neutral but count toward total

events_by_person = calendar_events_df.groupby('attendee_id').agg(
    total_appointments=('id', 'count'),
    completed_or_confirmed=('attendee_status', lambda x: x.isin(['COMPLETED', 'CONFIRMED']).sum()),
    canceled_or_noshow=('attendee_status', lambda x: x.isin(['CANCELED', 'NO_SHOW']).sum()),
).reset_index()

events_by_person['appointment_reliability_score'] = (
    events_by_person['completed_or_confirmed'] / events_by_person['total_appointments']
).round(3)

# --- Derive average_days_to_pay ---
# Join invoice_payments -> payment_log to get recordedat for each invoice's payments
# Then calculate days between invoice createdat and earliest payment recordedat

# Get invoice creation dates
invoice_dates = invoices_df[['id', 'personid', 'createdat']].copy()
invoice_dates['invoice_createdat'] = pd.to_datetime(invoice_dates['createdat'])

# Join invoice_payments with payment_log to get recordedat
payments_with_dates = invoice_payments_df.merge(
    payment_log_df[['id', 'recordedat']],
    left_on='paymentid',
    right_on='id',
    how='left'
)
payments_with_dates['payment_recordedat'] = pd.to_datetime(payments_with_dates['recordedat'])

# Get the first payment date per invoice (earliest recordedat)
first_payment_per_invoice = payments_with_dates.groupby('invoiceid').agg(
    first_payment_at=('payment_recordedat', 'min')
).reset_index()

# Join back to invoices to calculate days to pay
invoice_pay_times = invoice_dates.merge(
    first_payment_per_invoice,
    left_on='id',
    right_on='invoiceid',
    how='inner'
)
invoice_pay_times['days_to_pay'] = (
    invoice_pay_times['first_payment_at'] - invoice_pay_times['invoice_createdat']
).dt.days

# Average days to pay per person
avg_days_by_person = invoice_pay_times.groupby('personid').agg(
    average_days_to_pay=('days_to_pay', 'mean'),
    total_paid_invoices=('days_to_pay', 'count'),
).reset_index()
avg_days_by_person['average_days_to_pay'] = avg_days_by_person['average_days_to_pay'].round(1)

# --- Combine derived features into a single person-level dataframe ---
derived_features = persons_df[['id']].merge(
    events_by_person[['attendee_id', 'appointment_reliability_score', 'total_appointments']],
    left_on='id',
    right_on='attendee_id',
    how='left'
).merge(
    avg_days_by_person[['personid', 'average_days_to_pay', 'total_paid_invoices']],
    left_on='id',
    right_on='personid',
    how='left'
)

# Fill NaN for persons with no payment history (e.g. all invoices unpaid)
derived_features['average_days_to_pay'] = derived_features['average_days_to_pay'].fillna(-1)  # -1 = no payment history
derived_features['appointment_reliability_score'] = derived_features['appointment_reliability_score'].fillna(0.5)  # neutral default

# Keep only needed columns
derived_features = derived_features[['id', 'appointment_reliability_score', 'total_appointments', 'average_days_to_pay', 'total_paid_invoices']]

print(f"\u2705 Derived person-level features for {len(derived_features)} persons")
print(f"\nAppointment reliability score:")
print(f"   Mean: {derived_features['appointment_reliability_score'].mean():.3f}")
print(f"   Median: {derived_features['appointment_reliability_score'].median():.3f}")
print(f"   Range: {derived_features['appointment_reliability_score'].min():.3f} - {derived_features['appointment_reliability_score'].max():.3f}")
print(f"\nAverage days to pay:")
has_payments = derived_features[derived_features['average_days_to_pay'] > 0]
print(f"   Persons with payment history: {len(has_payments)}/{len(derived_features)}")
print(f"   Mean: {has_payments['average_days_to_pay'].mean():.1f} days")
print(f"   Median: {has_payments['average_days_to_pay'].median():.1f} days")
print(f"\nPreview:")
derived_features.head(10)

✅ Derived person-level features for 250 persons

Appointment reliability score:
   Mean: 0.740
   Median: 0.750
   Range: 0.000 - 1.000

Average days to pay:
   Persons with payment history: 250/250
   Mean: 11.6 days
   Median: 11.4 days

Preview:


,id,appointment_reliability_score,total_appointments,average_days_to_pay,total_paid_invoices
0,c375c13b-a867-4d36-bbb7-5feed52ef755,0.923,13,16.0,12
1,c73a705f-44b2-49ee-b8f7-a361133dc5de,0.833,6,7.5,4
2,198d34a5-a072-4871-800d-4a1b5df61b51,0.857,14,9.3,13
3,d7894388-84cd-4cb8-a87c-cffe7e63e95e,0.583,12,11.3,7
4,6185fa1f-6858-4a48-95d5-306ab4098e70,0.750,12,20.5,4
5,5238a578-a1f0-46cc-b1e6-da1df4de16cf,0.273,22,8.8,11
6,0aa1318c-2e81-43ac-9efa-2d7375413df4,0.583,12,5.2,12
7,18826775-784f-4f50-9b0e-c15bba885ec7,0.875,8,14.0,8
8,9243abc5-d06c-428a-899b-faadb0d39c4e,0.600,5,17.2,13
9,61f6c469-6976-41d9-875c-51f366413bbe,1.000,4,18.8,4


## Step 3: Enrich Persons with Census Data + Derived Features
Merges persons with their addresses to get `postal_code`, then joins with census data on zip code to attach income and demographic features. Also joins the derived behavioral features from Step 2.

In [3]:
# Load census data
census_path = '../../data/processed/census_zip_features_model_ready.csv'
census_df = pd.read_csv(census_path, dtype={'zip_code': str})

# Only use zip codes that have complete, valid census data
valid_census = census_df[
    (census_df['census_match'] == 1) &
    (census_df['median_household_income'].notna()) &
    (census_df['poverty_rate_pct'].notna()) &
    (census_df['poverty_rate_pct'] >= 0) &
    (census_df['unemployment_rate_pct'] >= 0) &
    (census_df['average_household_size'].notna()) &
    (census_df['average_household_size'] > 0)
].copy()

print(f"Census zip codes total: {len(census_df)}")
print(f"Valid zip codes with complete data: {len(valid_census)}")

# Merge persons with their addresses to get postal_code
persons_with_zip = persons_df[['id', 'is_guardian', 'entry_date']].merge(
    addresses_df[['person_id', 'postal_code']],
    left_on='id',
    right_on='person_id',
    how='left'
)

# Join with census data on zip code
persons_enriched = persons_with_zip.merge(
    valid_census[['zip_code', 'median_household_income', 'mean_household_income',
                  'poverty_rate_pct', 'unemployment_rate_pct',
                  'private_insurance_coverage_pct', 'public_insurance_coverage_pct',
                  'average_household_size', 'bachelors_or_higher_pct']],
    left_on='postal_code',
    right_on='zip_code',
    how='left'
)

# Join with derived features (appointment_reliability_score, average_days_to_pay)
persons_enriched = persons_enriched.merge(
    derived_features[['id', 'appointment_reliability_score', 'average_days_to_pay']],
    on='id',
    how='left'
)

# Verify join quality
census_matched = persons_enriched['median_household_income'].notna().sum()
print(f"\n\u2705 Census + derived features join complete")
print(f"   Persons with census data: {census_matched}/{len(persons_enriched)} ({census_matched/len(persons_enriched)*100:.1f}%)")
print(f"   Median household income range: ${persons_enriched['median_household_income'].min():,.0f} - ${persons_enriched['median_household_income'].max():,.0f}")
print(f"   Mean poverty rate: {persons_enriched['poverty_rate_pct'].mean():.1f}%")
print(f"   Appointment reliability (mean): {persons_enriched['appointment_reliability_score'].mean():.3f}")
print(f"   Avg days to pay (mean, excl. no-history): {persons_enriched[persons_enriched['average_days_to_pay'] > 0]['average_days_to_pay'].mean():.1f}")
print(f"\nPreview:")
persons_enriched[['id', 'postal_code', 'median_household_income', 'appointment_reliability_score', 'average_days_to_pay']].head()

Census zip codes total: 37963
Valid zip codes with complete data: 30506

✅ Census + derived features join complete
   Persons with census data: 250/250 (100.0%)
   Median household income range: $2,499 - $220,746
   Mean poverty rate: 12.7%
   Appointment reliability (mean): 0.740
   Avg days to pay (mean, excl. no-history): 11.6

Preview:


,id,postal_code,median_household_income,appointment_reliability_score,average_days_to_pay
0,c375c13b-a867-4d36-bbb7-5feed52ef755,75831,52188.0,0.923,16.0
1,c73a705f-44b2-49ee-b8f7-a361133dc5de,63645,57862.0,0.833,7.5
2,198d34a5-a072-4871-800d-4a1b5df61b51,05055,140590.0,0.857,9.3
3,d7894388-84cd-4cb8-a87c-cffe7e63e95e,39827,54976.0,0.583,11.3
4,6185fa1f-6858-4a48-95d5-306ab4098e70,77011,49163.0,0.750,20.5


## Step 4: Assemble Flat Training Table
Joins invoices with the enriched person data and derives time-based features from invoice timestamps. Produces the final flat table used to train the ML model.

**Feature groups:**
- **Invoice:** `amount`, `created_day_of_week`, `created_day_of_month`, `created_hour`, `origin`, `surchargingenabled`
- **Person:** `is_guardian`, `account_age_days`, `tenure_months`, `is_new_patient`
- **Derived behavioral:** `average_days_to_pay`, `appointment_reliability_score`
- **Census:** `median_household_income`, `poverty_rate_pct`, `average_household_size`, `bachelors_or_higher_pct`, `unemployment_rate_pct`

**Target:** `paid_within_30_days` = 1 if first payment was made within 30 calendar days of invoice creation, else 0 (includes unpaid)

In [4]:
# Merge invoices with enriched person data
flat_df = invoices_df.merge(
    persons_enriched,
    left_on='personid',
    right_on='id',
    how='left',
    suffixes=('', '_person')
)

# --- Derive features ---

# Time-based features from invoice creation date
flat_df['createdat_ts'] = pd.to_datetime(flat_df['createdat'])
flat_df['created_day_of_week'] = flat_df['createdat_ts'].dt.dayofweek
flat_df['created_day_of_month'] = flat_df['createdat_ts'].dt.day  # "Payday" feature: bills near 1st/15th pay faster
flat_df['created_hour'] = flat_df['createdat_ts'].dt.hour

# Account age: days between person entry_date and invoice creation
flat_df['entry_date_ts'] = pd.to_datetime(flat_df['entry_date'])
flat_df['account_age_days'] = (flat_df['createdat_ts'] - flat_df['entry_date_ts']).dt.days

# Tenure in months
flat_df['tenure_months'] = flat_df['account_age_days'] // 30

# "Cold Start" flag: is_new_patient = 1 if no prior payment history
flat_df['is_new_patient'] = (flat_df['average_days_to_pay'] == -1).astype(int)

# --- Target variable: paid_within_30_days ---
# 1 if the first payment was recorded within 30 calendar days of invoice creation, else 0
# This is stricter than "was ever paid" — unpaid and late-paid invoices both get 0
flat_df = flat_df.merge(
    invoice_pay_times[['id', 'days_to_pay']].rename(columns={'id': 'invoice_id_pay'}),
    left_on='id',
    right_on='invoice_id_pay',
    how='left'
)
flat_df['paid_within_30_days'] = (
    (flat_df['days_to_pay'].notna()) & (flat_df['days_to_pay'] <= 30)
).astype(int)

# Select final feature columns + target
feature_columns = [
    # Invoice features
    'amount',
    'created_day_of_week',
    'created_day_of_month',
    'created_hour',
    'origin',
    'surchargingenabled',
    # Person features
    'is_guardian',
    'account_age_days',
    'tenure_months',
    'is_new_patient',
    # Derived behavioral features (from payment_log and calendar_events)
    'average_days_to_pay',
    'appointment_reliability_score',
    # Census features
    'median_household_income',
    'poverty_rate_pct',
    'average_household_size',
    'bachelors_or_higher_pct',
    'unemployment_rate_pct',
]

# Keep IDs for reference + features + target
output_columns = ['personid', 'postal_code'] + feature_columns + ['status', 'paid_within_30_days']
training_df = flat_df[output_columns].copy()

# Convert booleans to int for model compatibility
bool_cols = ['is_guardian', 'surchargingenabled']
for col in bool_cols:
    training_df[col] = training_df[col].astype(int)

print(f"✅ Flat training table assembled: {training_df.shape[0]} rows x {training_df.shape[1]} columns")
print(f"\nTarget distribution (paid_within_30_days):")
print(f"   Paid within 30d (1): {training_df['paid_within_30_days'].sum()} ({training_df['paid_within_30_days'].mean()*100:.1f}%)")
print(f"   Late or unpaid  (0): {(training_df['paid_within_30_days'] == 0).sum()} ({(1 - training_df['paid_within_30_days'].mean())*100:.1f}%)")
print(f"\nNew features:")
print(f"   is_new_patient: {training_df['is_new_patient'].sum()} ({training_df['is_new_patient'].mean()*100:.1f}%)")
print(f"   created_day_of_month range: {training_df['created_day_of_month'].min()}-{training_df['created_day_of_month'].max()}")
print(f"\nFeature columns ({len(feature_columns)}):")
for col in feature_columns:
    print(f"   {col}")
print(f"\nNull check:")
print(training_df[feature_columns].isnull().sum().to_string())
print(f"\nPreview:")
training_df.head()

✅ Flat training table assembled: 4000 rows x 21 columns

Target distribution (paid_within_30_days):
   Paid within 30d (1): 2419 (60.5%)
   Late or unpaid  (0): 1581 (39.5%)

New features:
   is_new_patient: 0 (0.0%)
   created_day_of_month range: 1-31

Feature columns (17):
   amount
   created_day_of_week
   created_day_of_month
   created_hour
   origin
   surchargingenabled
   is_guardian
   account_age_days
   tenure_months
   is_new_patient
   average_days_to_pay
   appointment_reliability_score
   median_household_income
   poverty_rate_pct
   average_household_size
   bachelors_or_higher_pct
   unemployment_rate_pct

Null check:
amount                           0
created_day_of_week              0
created_day_of_month             0
created_hour                     0
origin                           0
surchargingenabled               0
is_guardian                      0
account_age_days                 0
tenure_months                    0
is_new_patient                   0
avera

,personid,postal_code,amount,created_day_of_week,created_day_of_month,created_hour,origin,surchargingenabled,is_guardian,account_age_days,...,is_new_patient,average_days_to_pay,appointment_reliability_score,median_household_income,poverty_rate_pct,average_household_size,bachelors_or_higher_pct,unemployment_rate_pct,status,paid_within_30_days
0,be047312-e14a-4dc2-b757-2c13385a7051,61477,10078,2,25,20,6,0,0,30,...,0,12.1,0.500,60227.0,17.9,2.52,15.3,17.8,6,0
1,354b7f4a-2ca0-40aa-8b24-2df67f039756,32073,204532,2,17,10,3,0,0,54,...,0,5.0,0.889,76139.0,9.5,2.55,27.4,2.4,6,0
2,61917d19-c507-483c-8d82-0d4f18580b02,80651,125982,4,16,20,2,0,1,141,...,0,10.7,0.625,88608.0,7.3,2.78,23.1,4.8,6,0
3,ff081c8d-1d5e-478b-8448-a998c0863cd2,84063,239340,0,23,6,4,1,0,267,...,0,14.9,0.556,27050.0,28.3,2.32,4.0,0.0,6,0
4,7bb0fd61-f112-4422-9af1-fe8f229ada53,76301,161381,2,17,0,2,0,0,620,...,0,13.8,0.615,41828.0,28.9,2.36,12.6,4.0,6,0


## Step 5: Save Training Data
Saves the flat training table to `data/processed/training_data.csv`.

In [5]:
import os

output_dir = '../../data/processed'
os.makedirs(output_dir, exist_ok=True)

training_output = os.path.join(output_dir, 'training_data.csv')
training_df.to_csv(training_output, index=False)
print(f"\u2705 Saved flat training table: {training_output} ({len(training_df)} rows x {len(training_df.columns)} cols)")
print(f"\nSaved to {os.path.abspath(training_output)}")

✅ Saved flat training table: ../../data/processed/training_data.csv (4000 rows x 21 cols)

Saved to /Users/pablotrujillo/Development/hackathon/will-they-pay/data/processed/training_data.csv
